In [1]:
"""
仮想通貨 1分後 上昇/下落予測アプリ
起動方法: streamlit run app.py
"""

import time
import requests
import numpy as np
import pandas as pd
import streamlit as st
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

# ============================================================
#  ページ設定
# ============================================================
st.set_page_config(
    page_title="仮想通貨 予測ツール",
    page_icon="📈",
    layout="wide",
)

st.markdown("""
<style>
.big-price   { font-size: 2.8rem; font-weight: 700; }
.up-text     { color: #00c853; }
.down-text   { color: #ff1744; }
.neutral-text{ color: #90a4ae; }
.pred-box    { border-radius: 12px; padding: 1.2rem 1.5rem; text-align: center; }
.pred-up     { background: #003300; border: 2px solid #00c853; }
.pred-down   { background: #330000; border: 2px solid #ff1744; }
.pred-label  { font-size: 1.1rem; color: #b0bec5; margin-bottom: .3rem; }
.pred-result { font-size: 2rem; font-weight: 700; }
.metric-card { background: #1e1e2e; border-radius: 10px; padding: .8rem 1rem; }
</style>
""", unsafe_allow_html=True)

# ============================================================
#  定数
# ============================================================
COINS = {
    "Bitcoin (BTC)":  "bitcoin",
    "Ethereum (ETH)": "ethereum",
    "Solana (SOL)":   "solana",
    "BNB (BNB)":      "binancecoin",
    "XRP (XRP)":      "ripple",
}
COINGECKO_BASE = "https://api.coingecko.com/api/v3"

# ============================================================
#  データ取得関数
# ============================================================
@st.cache_data(ttl=60)
def fetch_ohlc(coin_id: str, days: int = 30) -> pd.DataFrame:
    """CoinGecko から OHLC を取得（1時間足）"""
    url = f"{COINGECKO_BASE}/coins/{coin_id}/ohlc"
    r = requests.get(url, params={"vs_currency": "usd", "days": days}, timeout=15)
    r.raise_for_status()
    df = pd.DataFrame(r.json(), columns=["ts", "open", "high", "low", "close"])
    df["ts"] = pd.to_datetime(df["ts"], unit="ms")
    df.set_index("ts", inplace=True)
    return df.sort_index()


@st.cache_data(ttl=30)
def fetch_current(coin_id: str) -> dict:
    """現在価格・24h変動率などを取得"""
    r = requests.get(
        f"{COINGECKO_BASE}/simple/price",
        params={
            "ids": coin_id,
            "vs_currencies": "usd",
            "include_24hr_change": "true",
            "include_market_cap": "true",
        },
        timeout=10,
    )
    r.raise_for_status()
    return r.json().get(coin_id, {})

# ============================================================
#  特徴量エンジニアリング
# ============================================================
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    # 移動平均
    for w in [3, 5, 10, 20]:
        d[f"ma{w}"] = d["close"].rolling(w).mean()

    # モメンタム
    for w in [1, 3, 5]:
        d[f"mom{w}"] = d["close"].pct_change(w)

    # ボラティリティ
    d["vol5"]  = d["close"].pct_change().rolling(5).std()
    d["vol10"] = d["close"].pct_change().rolling(10).std()

    # RSI
    delta = d["close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d["rsi"] = 100 - 100 / (1 + gain / loss)

    # ボリンジャーバンド位置
    mid   = d["close"].rolling(20).mean()
    std   = d["close"].rolling(20).std()
    d["bb_pos"] = (d["close"] - mid) / (2 * std + 1e-9)

    # MACD
    d["macd"] = (
        d["close"].ewm(span=12, adjust=False).mean()
        - d["close"].ewm(span=26, adjust=False).mean()
    )
    d["macd_signal"] = d["macd"].ewm(span=9, adjust=False).mean()
    d["macd_diff"]   = d["macd"] - d["macd_signal"]

    # 高値・安値レンジ
    d["hl_ratio"] = (d["high"] - d["low"]) / (d["close"] + 1e-9)

    # 目的変数: 1本後の終値が上がれば1
    d["target"] = (d["close"].shift(-1) > d["close"]).astype(int)

    return d


FEATURES = [
    "ma3","ma5","ma10","ma20",
    "mom1","mom3","mom5",
    "vol5","vol10",
    "rsi","bb_pos",
    "macd","macd_signal","macd_diff",
    "hl_ratio",
]

# ============================================================
#  モデル学習
# ============================================================
def train_model(df: pd.DataFrame):
    data = add_features(df).dropna()
    if len(data) < 50:
        return None, None, 0.0

    X = data[FEATURES].values
    y = data["target"].values

    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sc, y, test_size=0.2, shuffle=False
    )

    clf = RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    )
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    return clf, scaler, acc


# ============================================================
#  予測
# ============================================================
def predict_next(clf, scaler, df: pd.DataFrame):
    data   = add_features(df).dropna()
    latest = data[FEATURES].iloc[[-1]].values
    latest_sc = scaler.transform(latest)
    prob   = clf.predict_proba(latest_sc)[0]   # [P(下), P(上)]
    pred   = int(clf.predict(latest_sc)[0])
    return pred, prob[1]  # 予測ラベル, 上昇確率


# ============================================================
#  サイドバー
# ============================================================
st.sidebar.title("⚙️ 設定")
coin_label = st.sidebar.selectbox("通貨を選択", list(COINS.keys()))
coin_id    = COINS[coin_label]
days       = st.sidebar.slider("学習データ期間（日）", 7, 90, 30)
auto_refresh = st.sidebar.checkbox("自動更新（60秒ごと）", value=False)

st.sidebar.markdown("---")
st.sidebar.markdown("""
**使い方**
1. 通貨と期間を選ぶ
2. 「予測を実行」をクリック
3. 結果を参考に分析

**注意**
- 予測は統計的な参考値です
- 投資判断は自己責任でお願いします
- 短期予測の精度には限界があります
""")

# ============================================================
#  メイン
# ============================================================
st.title("📈 仮想通貨 1分後 予測ツール")
st.caption("CoinGecko API × ランダムフォレストによる価格方向予測")

# 現在価格表示
try:
    current = fetch_current(coin_id)
    price    = current.get("usd", 0)
    chg24    = current.get("usd_24h_change", 0)
    mcap     = current.get("usd_market_cap", 0)
    chg_color = "up-text" if chg24 >= 0 else "down-text"
    chg_sign  = "▲" if chg24 >= 0 else "▼"

    col1, col2, col3 = st.columns([2, 1, 1])
    with col1:
        st.markdown(
            f'<div class="big-price">${price:,.2f} '
            f'<span class="{chg_color}" style="font-size:1.4rem">'
            f'{chg_sign} {abs(chg24):.2f}%</span></div>',
            unsafe_allow_html=True,
        )
        st.caption(f"{coin_label} / USD  ｜  更新: {pd.Timestamp.now().strftime('%H:%M:%S')}")
    with col2:
        st.metric("時価総額", f"${mcap/1e9:.1f}B")
    with col3:
        st.metric("24H変動", f"{chg24:+.2f}%")

except Exception as e:
    st.error(f"価格取得エラー: {e}")
    st.stop()

st.divider()

# 学習 & 予測ボタン
col_btn, col_status = st.columns([1, 3])
run = col_btn.button("🔍 予測を実行", type="primary", use_container_width=True)

if run or auto_refresh:
    with st.spinner("データ取得・モデル学習中..."):
        try:
            df  = fetch_ohlc(coin_id, days)
            clf, scaler, acc = train_model(df)

            if clf is None:
                st.warning("データが少なすぎます。期間を長くしてください。")
                st.stop()

            pred, prob_up = predict_next(clf, scaler, df)
            prob_dn = 1 - prob_up

        except requests.RequestException as e:
            st.error(f"API エラー: {e}")
            st.stop()

    # ── 予測結果 ──
    st.subheader("🎯 予測結果")
    r1, r2, r3 = st.columns(3)

    if pred == 1:
        direction   = "📈 上昇"
        box_class   = "pred-up"
        dir_color   = "#00c853"
    else:
        direction   = "📉 下落"
        box_class   = "pred-down"
        dir_color   = "#ff1744"

    confidence = max(prob_up, prob_dn) * 100

    with r1:
        st.markdown(
            f'<div class="pred-box {box_class}">'
            f'<div class="pred-label">1本後の方向（参考）</div>'
            f'<div class="pred-result" style="color:{dir_color}">{direction}</div>'
            f'</div>',
            unsafe_allow_html=True,
        )
    with r2:
        st.markdown(
            f'<div class="pred-box" style="background:#1e1e2e;border:1px solid #37474f;border-radius:12px;padding:1.2rem 1.5rem;text-align:center">'
            f'<div class="pred-label">モデル信頼度</div>'
            f'<div class="pred-result" style="color:#ffca28">{confidence:.1f}%</div>'
            f'</div>',
            unsafe_allow_html=True,
        )
    with r3:
        st.markdown(
            f'<div class="pred-box" style="background:#1e1e2e;border:1px solid #37474f;border-radius:12px;padding:1.2rem 1.5rem;text-align:center">'
            f'<div class="pred-label">バックテスト精度</div>'
            f'<div class="pred-result" style="color:#40c4ff">{acc*100:.1f}%</div>'
            f'</div>',
            unsafe_allow_html=True,
        )

    # 確率バー
    st.markdown("#### 上昇 / 下落 確率")
    bar_col1, bar_col2 = st.columns(2)
    bar_col1.metric("📈 上昇確率", f"{prob_up*100:.1f}%")
    bar_col2.metric("📉 下落確率", f"{prob_dn*100:.1f}%")
    st.progress(float(prob_up))

    st.divider()

    # ── チャート ──
    st.subheader("📊 価格チャート")

    feat_df = add_features(df)

    chart_tab, rsi_tab, feat_tab = st.tabs(["価格 + MA", "RSI / MACD", "特徴量の重要度"])

    with chart_tab:
        chart_data = pd.DataFrame({
            "終値":   feat_df["close"],
            "MA5":    feat_df["ma5"],
            "MA20":   feat_df["ma20"],
        }).dropna()
        st.line_chart(chart_data, height=320)

    with rsi_tab:
        c1, c2 = st.columns(2)
        with c1:
            st.markdown("**RSI (14)**")
            rsi_df = feat_df["rsi"].dropna().to_frame()
            st.line_chart(rsi_df, height=220)
            latest_rsi = feat_df["rsi"].dropna().iloc[-1]
            if latest_rsi > 70:
                st.warning(f"RSI {latest_rsi:.1f} — 過買い圏（売られやすい）")
            elif latest_rsi < 30:
                st.success(f"RSI {latest_rsi:.1f} — 過売り圏（買われやすい）")
            else:
                st.info(f"RSI {latest_rsi:.1f} — 中立圏")
        with c2:
            st.markdown("**MACD**")
            macd_df = feat_df[["macd", "macd_signal"]].dropna()
            st.line_chart(macd_df, height=220)

    with feat_tab:
        importance = pd.Series(
            clf.feature_importances_, index=FEATURES
        ).sort_values(ascending=False)
        st.bar_chart(importance, height=300)
        st.caption("値が大きいほど予測に影響を与えている特徴量です")

    # ── 直近データ ──
    st.subheader("📋 直近データ")
    display_cols = ["open", "high", "low", "close", "rsi", "macd", "bb_pos"]
    st.dataframe(
        feat_df[display_cols].dropna().tail(20).round(4).iloc[::-1],
        use_container_width=True,
    )

    # 自動更新
    if auto_refresh:
        time.sleep(60)
        st.rerun()

else:
    st.info("👆 サイドバーで通貨を選んで「予測を実行」ボタンを押してください。")

    # 使い方ガイド
    st.markdown("""
    ### このツールについて

    **仕組み**
    - CoinGecko API からリアルタイムの価格データを取得
    - RSI・MACD・ボリンジャーバンド等 15種類の特徴量を計算
    - ランダムフォレストで「次の足が上がるか下がるか」を学習・予測

    **表示される指標**
    | 指標 | 説明 |
    |------|------|
    | 予測方向 | 1本後の価格が上昇か下落かの予測 |
    | モデル信頼度 | 予測の確からしさ（高いほど自信あり）|
    | バックテスト精度 | 過去データでの正解率 |
    | RSI | 過買い/過売りの判断（70超=過買い, 30未満=過売り）|
    | MACD | トレンドの方向・勢い |

    > ⚠️ このツールは教育・学習目的のサンプルです。予測は統計的な参考値であり、実際の投資判断には使用しないでください。
    """)

2026-04-13 06:17:49.518 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:17:49.519 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:17:49.634 
  command:

    streamlit run /home/vscode/.local/lib/python3.11/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-04-13 06:17:49.635 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:17:49.635 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-13 06:17:49.636 No runtime found, using MemoryCacheStorageManager
2026-04-13 06:17:49.637 No runtime found, using MemoryCacheStorageManager
2026-04-13 06:17:49.640 Thread 'MainThread': missing ScriptRunContext! This warn